In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import xgboost as xgb
import numpy as np
from matplotlib import pyplot as plt

In [2]:
df1 = pd.read_csv('default_of_credit_card_clients.csv')
df1

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000,1,3,1,39,0,0,0,0,0,0,188948,192815,208365,88004,31237,15980,8500,20000,5003,3047,5000,1000,0
29996,29997,150000,1,3,2,43,-1,-1,-1,-1,0,0,1683,1828,3502,8979,5190,0,1837,3526,8998,129,0,0,0
29997,29998,30000,1,2,2,37,4,3,2,-1,0,0,3565,3356,2758,20878,20582,19357,0,0,22000,4200,2000,3100,1
29998,29999,80000,1,3,1,41,1,-1,0,0,0,-1,-1645,78379,76304,52774,11855,48944,85900,3409,1178,1926,52964,1804,1


In [3]:
df2 = df1.astype(float)
df2

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1.0,20000.0,2.0,2.0,1.0,24.0,2.0,2.0,-1.0,-1.0,-2.0,-2.0,3913.0,3102.0,689.0,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1.0
1,2.0,120000.0,2.0,2.0,2.0,26.0,-1.0,2.0,0.0,0.0,0.0,2.0,2682.0,1725.0,2682.0,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1.0
2,3.0,90000.0,2.0,2.0,2.0,34.0,0.0,0.0,0.0,0.0,0.0,0.0,29239.0,14027.0,13559.0,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0.0
3,4.0,50000.0,2.0,2.0,1.0,37.0,0.0,0.0,0.0,0.0,0.0,0.0,46990.0,48233.0,49291.0,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0.0
4,5.0,50000.0,1.0,2.0,1.0,57.0,-1.0,0.0,-1.0,0.0,0.0,0.0,8617.0,5670.0,35835.0,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996.0,220000.0,1.0,3.0,1.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,188948.0,192815.0,208365.0,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0.0
29996,29997.0,150000.0,1.0,3.0,2.0,43.0,-1.0,-1.0,-1.0,-1.0,0.0,0.0,1683.0,1828.0,3502.0,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0.0
29997,29998.0,30000.0,1.0,2.0,2.0,37.0,4.0,3.0,2.0,-1.0,0.0,0.0,3565.0,3356.0,2758.0,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1.0
29998,29999.0,80000.0,1.0,3.0,1.0,41.0,1.0,-1.0,0.0,0.0,0.0,-1.0,-1645.0,78379.0,76304.0,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1.0


In [4]:
df2['util_avg_4'] = (((df2['BILL_AMT1']/df2['LIMIT_BAL'])+
                    (df2['BILL_AMT2']/df2['LIMIT_BAL'])+
                    (df2['BILL_AMT3']/df2['LIMIT_BAL'])+
                    (df2['BILL_AMT4']/df2['LIMIT_BAL']))/4).round(5)
df2['util_5'] = df2['BILL_AMT5']/df2['LIMIT_BAL'].round(5)
df2['util_6'] = df2['BILL_AMT6']/df2['LIMIT_BAL'].round(5)

In [5]:
df2['repay_avg_4'] = (((df2['PAY_AMT1']/df2['BILL_AMT1'])+
                   (df2['PAY_AMT2']/df2['BILL_AMT2'])+
                   (df2['PAY_AMT3']/df2['BILL_AMT3'])+
                   (df2['PAY_AMT4']/df2['BILL_AMT4']))/4).round(5)
df2['repay_5'] = df2['PAY_AMT5']/df2['BILL_AMT5'].round(5)
df2['repay_6'] = df2['PAY_AMT6']/df2['BILL_AMT6'].round(5)

In [6]:
df2

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month,util_avg_4,util_5,util_6,repay_avg_4,repay_5,repay_6
0,1.0,20000.0,2.0,2.0,1.0,24.0,2.0,2.0,-1.0,-1.0,-2.0,-2.0,3913.0,3102.0,689.0,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1.0,0.09630,0.000000,0.000000,NaN,NaN,NaN
1,2.0,120000.0,2.0,2.0,2.0,26.0,-1.0,2.0,0.0,0.0,0.0,2.0,2682.0,1725.0,2682.0,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1.0,0.02159,0.028792,0.027175,0.31455,0.000000,0.613309
2,3.0,90000.0,2.0,2.0,2.0,34.0,0.0,0.0,0.0,0.0,0.0,0.0,29239.0,14027.0,13559.0,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0.0,0.19766,0.166089,0.172767,0.07560,0.066899,0.321564
3,4.0,50000.0,2.0,2.0,1.0,37.0,0.0,0.0,0.0,0.0,0.0,0.0,46990.0,48233.0,49291.0,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0.0,0.86414,0.579180,0.590940,0.03690,0.036914,0.033844
4,5.0,50000.0,1.0,2.0,1.0,57.0,-1.0,0.0,-1.0,0.0,0.0,0.0,8617.0,5670.0,35835.0,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0.0,0.35531,0.382920,0.382620,1.85257,0.035987,0.035492
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996.0,220000.0,1.0,3.0,1.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,188948.0,192815.0,208365.0,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0.0,0.77060,0.141986,0.072636,0.05184,0.160067,0.062578
29996,29997.0,150000.0,1.0,3.0,2.0,43.0,-1.0,-1.0,-1.0,-1.0,0.0,0.0,1683.0,1828.0,3502.0,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0.0,0.02665,0.034600,0.000000,1.40104,0.000000,NaN
29997,29998.0,30000.0,1.0,2.0,2.0,37.0,4.0,3.0,2.0,-1.0,0.0,0.0,3565.0,3356.0,2758.0,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1.0,0.25464,0.686067,0.645233,2.04449,0.097172,0.160149
29998,29999.0,80000.0,1.0,3.0,1.0,41.0,1.0,-1.0,0.0,0.0,0.0,-1.0,-1645.0,78379.0,76304.0,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1.0,0.64316,0.148187,0.611800,-13.03085,4.467651,0.036858


In [7]:
df3 = pd.DataFrame()

In [8]:
df3['AGE']      = df2['AGE']
df3['PAY_0']    = df2['PAY_0']
df3['PAY_2']    = df2['PAY_2']
df3['PAY_3']    = df2['PAY_3']
df3['PAY_4']    = df2['PAY_4']
df3['PAY_5']    = df2['PAY_5']
df3['PAY_6']    = df2['PAY_6']
df3['util_avg_4'] = df2['util_avg_4']
df3['util_5']   = df2['util_5']
df3['util_6']   = df2['util_6']
df3['repay_avg_4'] = df2['repay_avg_4']
df3['repay_5']   = df2['repay_5']
df3['repay_6']   = df2['repay_6']
df3['default']   = df2['default payment next month']

In [9]:
df3

,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,util_avg_4,util_5,util_6,repay_avg_4,repay_5,repay_6,default
0,24.0,2.0,2.0,-1.0,-1.0,-2.0,-2.0,0.09630,0.000000,0.000000,NaN,NaN,NaN,1.0
1,26.0,-1.0,2.0,0.0,0.0,0.0,2.0,0.02159,0.028792,0.027175,0.31455,0.000000,0.613309,1.0
2,34.0,0.0,0.0,0.0,0.0,0.0,0.0,0.19766,0.166089,0.172767,0.07560,0.066899,0.321564,0.0
3,37.0,0.0,0.0,0.0,0.0,0.0,0.0,0.86414,0.579180,0.590940,0.03690,0.036914,0.033844,0.0
4,57.0,-1.0,0.0,-1.0,0.0,0.0,0.0,0.35531,0.382920,0.382620,1.85257,0.035987,0.035492,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.77060,0.141986,0.072636,0.05184,0.160067,0.062578,0.0
29996,43.0,-1.0,-1.0,-1.0,-1.0,0.0,0.0,0.02665,0.034600,0.000000,1.40104,0.000000,NaN,0.0
29997,37.0,4.0,3.0,2.0,-1.0,0.0,0.0,0.25464,0.686067,0.645233,2.04449,0.097172,0.160149,1.0
29998,41.0,1.0,-1.0,0.0,0.0,0.0,-1.0,0.64316,0.148187,0.611800,-13.03085,4.467651,0.036858,1.0


In [10]:
df3 = df3.fillna(0.0)
df3.isnull().sum()
df3

,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,util_avg_4,util_5,util_6,repay_avg_4,repay_5,repay_6,default
0,24.0,2.0,2.0,-1.0,-1.0,-2.0,-2.0,0.09630,0.000000,0.000000,0.00000,0.000000,0.000000,1.0
1,26.0,-1.0,2.0,0.0,0.0,0.0,2.0,0.02159,0.028792,0.027175,0.31455,0.000000,0.613309,1.0
2,34.0,0.0,0.0,0.0,0.0,0.0,0.0,0.19766,0.166089,0.172767,0.07560,0.066899,0.321564,0.0
3,37.0,0.0,0.0,0.0,0.0,0.0,0.0,0.86414,0.579180,0.590940,0.03690,0.036914,0.033844,0.0
4,57.0,-1.0,0.0,-1.0,0.0,0.0,0.0,0.35531,0.382920,0.382620,1.85257,0.035987,0.035492,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.77060,0.141986,0.072636,0.05184,0.160067,0.062578,0.0
29996,43.0,-1.0,-1.0,-1.0,-1.0,0.0,0.0,0.02665,0.034600,0.000000,1.40104,0.000000,0.000000,0.0
29997,37.0,4.0,3.0,2.0,-1.0,0.0,0.0,0.25464,0.686067,0.645233,2.04449,0.097172,0.160149,1.0
29998,41.0,1.0,-1.0,0.0,0.0,0.0,-1.0,0.64316,0.148187,0.611800,-13.03085,4.467651,0.036858,1.0


In [11]:
from sklearn.model_selection import train_test_split

In [12]:
X = df3.drop(['default'], axis='columns')
y = df3['default']
y

0        1.0
1        1.0
2        0.0
3        0.0
4        0.0
        ... 
29995    0.0
29996    0.0
29997    1.0
29998    1.0
29999    1.0
Name: default, Length: 30000, dtype: float64

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)
X_train.replace([np.inf, -np.inf], 0.0, inplace=True)
X_test.replace([np.inf, -np.inf], 0.0, inplace=True)
y_train.replace([np.inf, -np.inf], 0.0, inplace=True)
y_test.replace([np.inf, -np.inf], 0.0, inplace=True)

In [14]:
modelXGB = xgb.XGBClassifier(objective='binary:logistic')
modelXGB.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [15]:
modelXGB.score(X_test, y_test)

0.8116666666666666

In [16]:
modelXGB.predict

<bound method XGBClassifier.predict of XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)>